Every RAG system is built around three essential components:
- An embedding model that represents text as vectors
- A vector store that searches those vectors
- An LLM that generates answers from retrieved context <br>
The workflow has two main parts: data ingestion and query processing, which

#### data ingestion, you’ll follow these steps:
- Split documents into smaller chunks.
- Generate embeddings for each chunk.
- Store vectors in a vector database.

#### query processing, you’ll use a complete RAG pipeline
- Convert the user’s question into an embedding vector.
- Search the vector database for the closest matching vectors.
- Select the most relevant chunks.
- Send chunks together with the user’s question to the LLM.

In [ ]:
from IPython.display import Image, display
display(Image(filename="../images/1.png", width=500, height=500))

An OpenAI account and an API key from the OpenAI Platform website
Chroma for vector storage 

This recipe demonstrates the workflow by building a Harry Potter knowledge chatbot.below image the overall architecture this recipe implements. The retriever searches a Chroma vector store built from the Harry Potter books, and the generator uses OpenAI models to answer questions based on retrieved context.

In [ ]:
from IPython.display import Image, display
display(Image(filename="../images/2.png", width=500, height=500))

#### Step 1: Text chunking

In [ ]:
from pathlib import Path
import chromadb
from openai import OpenAI

def chunk_text(text, size=1000, overlap=200):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            bp = text.rfind("\n\n", start, end)
            if bp == -1:
                bp = text.rfind(". ", start, end)
            if bp > start:
                end = bp + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap if end < len(text) else end
    return chunks
file_path = Path("datasets/text_files/harry_potter_knowledge_base.txt")
text = file_path.read_text(encoding="utf-8")
chunks = chunk_text(text)

#### Step 2: Embed and store in Chroma

In [ ]:
client = OpenAI()
embedding_model = "text-embedding-3-small"

def embed_and_store(chunks, db_path, collection_name):
    chroma = chromadb.PersistentClient(path=str(db_path))
    collection = chroma.get_or_create_collection(
        name=collection_name,
        metadata={"description": "Harry Potter knowledge base"},
    )

    embeddings = []
    for i in range(0, len(chunks), 100):
        batch = chunks[i : i + 100]
        res = client.embeddings.create(model=embedding_model, input=batch)
        embeddings.extend([x.embedding for x in res.data])

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=embeddings,
        metadatas=[{"chunk_index": i} for i in range(len(chunks))],
    )
    return collection
chroma_db_dir = Path("chroma_db")
collection = embed_and_store(chunks, chroma_db_dir, "harry_potter_kb")

#### Step 3: Test retrieval

In [ ]:
def retrieve(question, top_k=3):
    q_emb = client.embeddings.create(
        model=embedding_model,
        input=question,
    ).data[0].embedding

    res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents"],
    )

    return res["documents"][0]


question = "Why did Uncle Vernon take the family to the hut by the sea?"
docs = retrieve(question)

### Step 4: Test generation

In [ ]:
def answer(question, docs):
    context = "\n\n---\n\n".join(docs)
    prompt = f"""Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:"""

    res = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
    )

    return res.choices[0].message.content


question = "Why did Uncle Vernon take the family to the hut by the sea?"
docs = retrieve(question)
answer_text = answer(question, docs)